# Docking Experiments

we need to do this twice once per preprep (Flare / Schroedinger)


In [4]:
!pip install ipywidgets loguru py3dmol rdkit 2>/dev/null | tail -n 1

In [5]:
import time
import os
import shutil
import json
import requests
import subprocess
from loguru import logger
import pandas as pd
from datetime import datetime

## check if docker is active

In [6]:
# define Diffdock NIM connection
base_url="http://127.0.0.1:8083/" 
query_url = base_url + "/molecular-docking/diffdock/generate"
health_check_url = base_url + "/v1/health/ready"

# check health of the DiffDock NIM endpoint
response = requests.get(health_check_url)
print(response.text)

true


## Overview of Experiment

In [7]:
# read overview for experiments
df_meta = pd.read_csv(r"../../../data/raw/docking/Experiment_Docking_Overview.csv")

df_meta.head(20)
#df_meta.tail(20)

,Ligand,Protein,Target_Name,Cofactors
0,BCS-CT37643,bcsin_226,pyr1-hab1,NaN
1,BCS-CV49374,bcsin_233,HPPD,FE2
2,BCS-CN76566,bcsin_235,pyr1-hab1,NaN
3,BCS-CY86357,bcsin_261,KRS1,LYS
4,BCS-CZ81034,bcsin_262,KRS1,LYS
5,BCS-CW64002,bcsin_300,DHODH,"FMN, ORO"
6,BCS-DA38023,bcsin_312,DHODH,"FMN, ORO"
7,BCS-DH93179,bcsin_349,DHODH,"FMN, ORO"
8,BCS-DM16159,bcsin_401,BCAT,PLP
9,BCS-CW64991,bcsin_404,ETF-DH,"SF4, FAD"


# Start Experiments

### Run 1 - Schroedinger Preped PDB + Standard SDf from SMILES

In [5]:
from nv_diffdock import run_diffdock_batch

# Define paths
csv_path = r"../../../data/raw/docking/Experiment_Docking_Overview.csv"  # CSV with columns: ligand_name, protein_name
ligand_dir = "../../../data/raw/docking/ligands_3d"  # Directory with ligand SDF files - "../../data/raw/docking/ligands_3d" = the ligands from smiles "../../../data/interim/docking/ligands_from_prep_Flare_SDF" = Ligands from prep
protein_dir = "../../../data/interim/docking/prep_Schroe"  # Directory with protein PDB files

# Output directory--> CHANGE PER EXPERIMENTAL RUN
output_dir = "../../../data/interim/docking/Res_Diffdock_Schroe_R5"


In [7]:
# Run the batch inference 

results = run_diffdock_batch(
    csv_path=csv_path,
    ligand_dir=ligand_dir,
    protein_dir=protein_dir,
    output_base_dir=output_dir,
    query_url=query_url,
    ligand_col='Ligand',  # Customize column names if needed
    protein_col='Protein', # Customize column names if needed
    num_poses=10,
    steps=18, # reverse diffusion steps - per paper set to 20 --> for diffdock 2 reduced to 18 as neede by container 
)

print(f"Processing completed with {results['successful']} successful and {results['failed']} failed runs.")

2025-03-31 13:42:01.093 | INFO     | nv_diffdock:run_diffdock_batch:118 - Loaded 13 ligand-protein combinations from ../../../data/raw/docking/Experiment_Docking_Overview.csv
2025-03-31 13:42:01.094 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 1/13: BCS-CT37643 with bcsin_226
2025-03-31 13:42:07.732 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-CT37643 with bcsin_226 took 6.64 seconds
2025-03-31 13:42:07.735 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 2/13: BCS-CV49374 with bcsin_233


Request successful, output saved to output.json


2025-03-31 13:42:09.979 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-CV49374 with bcsin_233 took 2.24 seconds
2025-03-31 13:42:09.983 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 3/13: BCS-CN76566 with bcsin_235


Request successful, output saved to output.json


2025-03-31 13:42:12.780 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-CN76566 with bcsin_235 took 2.80 seconds
2025-03-31 13:42:12.783 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 4/13: BCS-CY86357 with bcsin_261


Request successful, output saved to output.json


... (16 outputs left out)

2025-03-31 13:42:48.364 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-DR80164 with bcsin_437 took 3.07 seconds
2025-03-31 13:42:48.369 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 13/13: BCS-DK97290 with bcsin_442


Request successful, output saved to output.json


2025-03-31 13:42:50.988 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-DK97290 with bcsin_442 took 2.62 seconds
2025-03-31 13:42:50.992 | INFO     | nv_diffdock:run_diffdock_batch:237 - Processing complete. Successful: 13, Failed: 0
2025-03-31 13:42:50.993 | INFO     | nv_diffdock:run_diffdock_batch:238 - Summary saved to ../../../data/interim/docking/Res_Diffdock_Schroe_R5/processing_summary.json


Request successful, output saved to output.json
Processing completed with 13 successful and 0 failed runs.


### Run 2 - Flare Preped PDB + Standard SDF from SMILES

In [8]:
from nv_diffdock import run_diffdock_batch

# Define paths
csv_path = r"../../../data/raw/docking/Experiment_Docking_Overview.csv"  # CSV with columns: ligand_name, protein_name
ligand_dir = "../../../data/raw/docking/ligands_3d"  # Directory with ligand SDF files - "../../data/raw/docking/ligands_3d" = the ligands from smiles "../../../data/interim/docking/ligands_from_prep_Flare_SDF" = Ligands from prep
protein_dir = "../../../data/interim/docking/prep_Flare"  # Directory with protein PDB files

# Output directory--> CHANGE PER EXPERIMENTAL RUN
output_dir = "../../../data/interim/docking/Res_Diffdock_Flare_R6"


In [9]:
# Run the batch inference 

results = run_diffdock_batch(
    csv_path=csv_path,
    ligand_dir=ligand_dir,
    protein_dir=protein_dir,
    output_base_dir=output_dir,
    query_url=query_url,
    ligand_col='Ligand',  # Customize column names if needed
    protein_col='Protein', # Customize column names if needed
    num_poses=10,
    steps=18, # reverse diffusion steps - per paper set to 20
)

print(f"Processing completed with {results['successful']} successful and {results['failed']} failed runs.")

2025-03-31 13:42:58.933 | INFO     | nv_diffdock:run_diffdock_batch:118 - Loaded 13 ligand-protein combinations from ../../../data/raw/docking/Experiment_Docking_Overview.csv
2025-03-31 13:42:58.934 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 1/13: BCS-CT37643 with bcsin_226
2025-03-31 13:43:01.185 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-CT37643 with bcsin_226 took 2.25 seconds
2025-03-31 13:43:01.189 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 2/13: BCS-CV49374 with bcsin_233


Request successful, output saved to output.json


2025-03-31 13:43:03.719 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-CV49374 with bcsin_233 took 2.53 seconds
2025-03-31 13:43:03.722 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 3/13: BCS-CN76566 with bcsin_235


Request successful, output saved to output.json


2025-03-31 13:43:06.538 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-CN76566 with bcsin_235 took 2.82 seconds
2025-03-31 13:43:06.542 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 4/13: BCS-CY86357 with bcsin_261


Request successful, output saved to output.json


... (16 outputs left out)

2025-03-31 13:43:41.822 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-DR80164 with bcsin_437 took 2.86 seconds
2025-03-31 13:43:41.826 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 13/13: BCS-DK97290 with bcsin_442


Request successful, output saved to output.json


2025-03-31 13:43:44.504 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-DK97290 with bcsin_442 took 2.68 seconds
2025-03-31 13:43:44.507 | INFO     | nv_diffdock:run_diffdock_batch:237 - Processing complete. Successful: 13, Failed: 0
2025-03-31 13:43:44.508 | INFO     | nv_diffdock:run_diffdock_batch:238 - Summary saved to ../../../data/interim/docking/Res_Diffdock_Flare_R6/processing_summary.json


Request successful, output saved to output.json
Processing completed with 13 successful and 0 failed runs.


### Run 3 - Schroedinger Preped PDB + Schroe Preped Ligand 

In [10]:
from nv_diffdock import run_diffdock_batch

# Define paths
csv_path = r"../../../data/raw/docking/Experiment_Docking_Overview.csv"  # CSV with columns: ligand_name, protein_name
ligand_dir = "../../../data/interim/docking/ligands_from_prep_Schroe_SDF"  # Directory with ligand SDF files - "../../data/raw/docking/ligands_3d" = the ligands from smiles "../../../data/interim/docking/ligands_from_prep_Flare_SDF" = Ligands from prep
protein_dir = "../../../data/interim/docking/prep_Schroe"  # Directory with protein PDB files

# Output directory--> CHANGE PER EXPERIMENTAL RUN
output_dir = "../../../data/interim/docking/Res_Diffdock_Schroe_R7"

In [11]:
# Run the batch inference 

results = run_diffdock_batch(
    csv_path=csv_path,
    ligand_dir=ligand_dir,
    protein_dir=protein_dir,
    output_base_dir=output_dir,
    query_url=query_url,
    ligand_col='Ligand',  # Customize column names if needed
    protein_col='Protein', # Customize column names if needed
    num_poses=10,
    steps=18, # reverse diffusion steps - per paper set to 20
)

print(f"Processing completed with {results['successful']} successful and {results['failed']} failed runs.")

2025-03-31 13:44:32.736 | INFO     | nv_diffdock:run_diffdock_batch:118 - Loaded 13 ligand-protein combinations from ../../../data/raw/docking/Experiment_Docking_Overview.csv
2025-03-31 13:44:32.737 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 1/13: BCS-CT37643 with bcsin_226
2025-03-31 13:44:35.275 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-CT37643 with bcsin_226 took 2.54 seconds
2025-03-31 13:44:35.278 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 2/13: BCS-CV49374 with bcsin_233


Request successful, output saved to output.json


2025-03-31 13:44:37.519 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-CV49374 with bcsin_233 took 2.24 seconds
2025-03-31 13:44:37.522 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 3/13: BCS-CN76566 with bcsin_235


Request successful, output saved to output.json


2025-03-31 13:44:40.349 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-CN76566 with bcsin_235 took 2.83 seconds
2025-03-31 13:44:40.352 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 4/13: BCS-CY86357 with bcsin_261


Request successful, output saved to output.json


... (14 outputs left out)

2025-03-31 13:45:44.275 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-DM58319 with bcsin_434 took 13.14 seconds
2025-03-31 13:45:44.283 | ERROR    | nv_diffdock:run_diffdock_batch:155 - Ligand file not found: ../../../data/interim/docking/ligands_from_prep_Schroe_SDF/BCS-DR80164.sdf
2025-03-31 13:45:44.284 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 13/13: BCS-DK97290 with bcsin_442


Request successful, output saved to output.json


2025-03-31 13:45:47.220 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-DK97290 with bcsin_442 took 2.94 seconds
2025-03-31 13:45:47.223 | INFO     | nv_diffdock:run_diffdock_batch:237 - Processing complete. Successful: 12, Failed: 1
2025-03-31 13:45:47.224 | INFO     | nv_diffdock:run_diffdock_batch:238 - Summary saved to ../../../data/interim/docking/Res_Diffdock_Schroe_R7/processing_summary.json


Request successful, output saved to output.json
Processing completed with 12 successful and 1 failed runs.


### Run 4 - Flare Preped PDB + Flare Preped Ligand

In [13]:
from nv_diffdock import run_diffdock_batch

# Define paths
csv_path = r"../../../data/raw/docking/Experiment_Docking_Overview.csv"  # CSV with columns: ligand_name, protein_name
ligand_dir = "../../../data/interim/docking/ligands_from_prep_Flare_SDF"  # Directory with ligand SDF files - "../../data/raw/docking/ligands_3d" = the ligands from smiles "../../../data/interim/docking/ligands_from_prep_Flare_SDF" = Ligands from prep
protein_dir = "../../../data/interim/docking/prep_Flare"  # Directory with protein PDB files

# Output directory--> CHANGE PER EXPERIMENTAL RUN
output_dir = "../../../data/interim/docking/Res_Diffdock_Flare_R8"


In [22]:
# Run the batch inference 

results = run_diffdock_batch(
    csv_path=csv_path,
    ligand_dir=ligand_dir,
    protein_dir=protein_dir,
    output_base_dir=output_dir,
    query_url=query_url,
    ligand_col='Ligand',  # Customize column names if needed
    protein_col='Protein', # Customize column names if needed
    num_poses=10,
    steps=20, # reverse diffusion steps - per paper set to 20
)

print(f"Processing completed with {results['successful']} successful and {results['failed']} failed runs.")

2025-03-31 11:46:38.081 | INFO     | nv_diffdock:run_diffdock_batch:118 - Loaded 13 ligand-protein combinations from ../../../data/raw/docking/Experiment_Docking_Overview.csv
2025-03-31 11:46:38.082 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 1/13: BCS-CT37643 with bcsin_226
2025-03-31 11:46:40.515 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-CT37643 with bcsin_226 took 2.43 seconds
2025-03-31 11:46:40.519 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 2/13: BCS-CV49374 with bcsin_233


Request successful, output saved to output.json


2025-03-31 11:46:42.952 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-CV49374 with bcsin_233 took 2.43 seconds
2025-03-31 11:46:42.955 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 3/13: BCS-CN76566 with bcsin_235


Request successful, output saved to output.json


2025-03-31 11:46:45.962 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-CN76566 with bcsin_235 took 3.01 seconds
2025-03-31 11:46:45.965 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 4/13: BCS-CY86357 with bcsin_261


Request successful, output saved to output.json


... (14 outputs left out)

2025-03-31 11:47:50.398 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-DM58319 with bcsin_434 took 12.08 seconds
2025-03-31 11:47:50.405 | ERROR    | nv_diffdock:run_diffdock_batch:155 - Ligand file not found: ../../../data/interim/docking/ligands_from_prep_Flare_SDF/BCS-DR80164.sdf
2025-03-31 11:47:50.406 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 13/13: BCS-DK97290 with bcsin_442


Request successful, output saved to output.json


2025-03-31 11:47:53.256 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-DK97290 with bcsin_442 took 2.85 seconds
2025-03-31 11:47:53.259 | INFO     | nv_diffdock:run_diffdock_batch:237 - Processing complete. Successful: 12, Failed: 1
2025-03-31 11:47:53.260 | INFO     | nv_diffdock:run_diffdock_batch:238 - Summary saved to ../../../data/interim/docking/Res_Diffdock_Flare_R4/processing_summary.json


Request successful, output saved to output.json
Processing completed with 12 successful and 1 failed runs.


# AF Fold Run

In [8]:
from nv_diffdock import run_diffdock_batch

# Define paths
csv_path = r"../../../data/raw/docking/Experiment_Docking_Overview.csv"  # CSV with columns: ligand_name, protein_name
ligand_dir = "../../../data/raw/docking/ligands_3d"  # Directory with ligand SDF files - "../../data/raw/docking/ligands_3d" = the ligands from smiles "../../../data/interim/docking/ligands_from_prep_Flare_SDF" = Ligands from prep
protein_dir = "../../../data/interim/APOs"  # Directory with protein PDB files

# Output directory--> CHANGE PER EXPERIMENTAL RUN
output_dir = "../../../data/interim/docking/Res_Diffdock_AF"

In [9]:
# Run the batch inference 

results = run_diffdock_batch(
    csv_path=csv_path,
    ligand_dir=ligand_dir,
    protein_dir=protein_dir,
    output_base_dir=output_dir,
    query_url=query_url,
    ligand_col='Ligand',  # Customize column names if needed
    protein_col='Protein', # Customize column names if needed
    num_poses=10,
    steps=18, # reverse diffusion steps - per paper set to 20 --> for diffdock 2 reduced to 18 as needed by container 
)

print(f"Processing completed with {results['successful']} successful and {results['failed']} failed runs.")

2025-04-11 06:28:14.142 | INFO     | nv_diffdock:run_diffdock_batch:118 - Loaded 13 ligand-protein combinations from ../../../data/raw/docking/Experiment_Docking_Overview.csv
2025-04-11 06:28:14.143 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 1/13: BCS-CT37643 with bcsin_226
2025-04-11 06:29:07.687 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-CT37643 with bcsin_226 took 53.54 seconds
2025-04-11 06:29:07.689 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 2/13: BCS-CV49374 with bcsin_233


Request successful, output saved to output.json


2025-04-11 06:29:10.036 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-CV49374 with bcsin_233 took 2.35 seconds
2025-04-11 06:29:10.039 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 3/13: BCS-CN76566 with bcsin_235


Request successful, output saved to output.json


2025-04-11 06:29:11.503 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-CN76566 with bcsin_235 took 1.46 seconds
2025-04-11 06:29:11.505 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 4/13: BCS-CY86357 with bcsin_261


Request successful, output saved to output.json


... (16 outputs left out)

2025-04-11 06:29:36.772 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-DR80164 with bcsin_437 took 1.70 seconds
2025-04-11 06:29:36.774 | INFO     | nv_diffdock:run_diffdock_batch:180 - Processing 13/13: BCS-DK97290 with bcsin_442


Request successful, output saved to output.json


2025-04-11 06:29:39.270 | DEBUG    | nv_diffdock:run_diffdock_batch:201 - BCS-DK97290 with bcsin_442 took 2.50 seconds
2025-04-11 06:29:39.272 | INFO     | nv_diffdock:run_diffdock_batch:237 - Processing complete. Successful: 13, Failed: 0
2025-04-11 06:29:39.273 | INFO     | nv_diffdock:run_diffdock_batch:238 - Summary saved to ../../../data/interim/docking/Res_Diffdock_AF/processing_summary.json


Request successful, output saved to output.json
Processing completed with 13 successful and 0 failed runs.
